# Homography test

## Imports

In [1]:
from sources.baseline_detection import BaselineDetection
from sources.scale import Scale
import pandas as pd
import numpy as np

## Description

- Define start-point and end-point of the movement
- Initialize the BaselineDetection (homography componenent)
- Calculate distance between the two points
- True: Report the distance in the distance file of the video; False: Rework the homography system implementation

### Define points

In [2]:
def retrieve_data(coordinates_path):
    # load the data into a pandas DataFrame
    df = pd.read_csv(coordinates_path)

    # define last frame number
    max_frame = df["frame"].max()

    # first position of the movement
    start_pt_1 = df[(df["frame"] == 0) & (df["id"] == 1)]
    start_pt_2 = df[(df["frame"] == 0) & (df["id"] == 2)]

    # last position of the movement
    end_pt_1 = df[(df["frame"] == max_frame) & (df["id"] == 1)]
    end_pt_2 = df[(df["frame"] == max_frame) & (df["id"] == 2)]

    return start_pt_1, start_pt_2, end_pt_1, end_pt_2

### Calculate distance between the two points

In [3]:
def apply_homography(pt, h_matrix):
    point = np.array([pt[0], pt[1], 1.0])
    transformed = h_matrix @ point

    return transformed[:2] / transformed[2]

In [4]:
def convert_bounding_box_to_point(bounding_box):
    x1 = bounding_box["x1"].values[0]
    x2 = bounding_box["x2"].values[0]
    x = x1 + ((x2 - x1) / 2)

    y1 = bounding_box["y1"].values[0]
    y2 = bounding_box["y2"].values[0]
    y = y1 + ((y2 - y1) / 2)

    return (x, y)

In [5]:
def calculate_distance(points: list, h: np.ndarray, scale: float):
    # scala = Scale("../data/data/points_cropped_schema.json")
    # print(f"Scale factor being used: {scala.scale}")

    total_distance = 0

    for i in range(len(points) - 1):
        start_pt = points[i]
        end_pt = points[i + 1]

        # apply homography
        new_start_pt = apply_homography(start_pt, h)
        # print(f"frame: {start_pt}")
        # print(f"schema: {new_start_pt}")

        new_end_pt = apply_homography(end_pt, h)
        # print(f"frame: {end_pt}")
        # print(f"schema: {new_end_pt}")

        # # calculate distance between new_start_pt and new_end_pt
        # distance_between_points = np.sqrt(
        #     (new_end_pt[0] - new_start_pt[0]) ** 2 +
        #     (new_end_pt[1] - new_start_pt[1]) ** 2
        # )

        # scale_factor = 28.0 / distance_between_points
        # print(f"factor: {scale_factor}")

        # calculate distance
        distance = np.sqrt(
            (new_end_pt[0] - new_start_pt[0]) ** 2 +
            (new_end_pt[1] - new_start_pt[1]) ** 2
        ) * scale

        print(f"Distance between point {i} and point {i + 1}: {distance} meters")

        total_distance += distance

    print(f"Total distance: {total_distance}")

In [6]:
scale = Scale("../data/data/points_cropped_schema.json")

## Use Cases

### eval.mov

In [7]:
# initialize component for the homography
bd = BaselineDetection(
    frame_points_path="../data/data/eval_points.json",
    schema_points_path="../data/data/points_cropped_schema.json"
)
h, inv_h = bd.calculate_homography()

bd.assess_homography_quality(h)


start_pt_1 = (195, 582)
end_pt_1 = (1666, 1494) # Distance: 28.0m

start_pt_2 = (2032, 539)
end_pt_2 = (553, 811) # Distance: 15.0m


calculate_distance([start_pt_1, start_pt_2], h, scale.scale) # 28.0

print()

calculate_distance([end_pt_1, end_pt_2], h, scale.scale) # 15.0


Point 0: Error = 16.79
Point 1: Error = 0.73
Point 2: Error = 16.31
Point 3: Error = 1.01
Point 4: Error = 8.95
Point 5: Error = 29.40
Point 6: Error = 28.49
Point 7: Error = 38.54
Point 8: Error = 0.72
Point 9: Error = 0.54
Point 10: Error = 0.37
Point 11: Error = 0.37
Point 12: Error = 0.32
Point 13: Error = 0.67
Point 14: Error = 0.58
Point 15: Error = 0.68
Point 16: Error = 13.89
Point 17: Error = 13.12
Average reprojection error: 9.53
Distance between point 0 and point 1: 21.42547717372008 meters
Total distance: 21.42547717372008

Distance between point 0 and point 1: 14.523720039968211 meters
Total distance: 14.523720039968211


#### Case 1: 28.0 meters with 10 points

![Image](../../assets/homography_test_1.png)

In [8]:
# example: calculate distance of a trajectory with more points
test = [
    (297, 579),
    (355, 612),
    (453, 666),
    (568, 722),
    (679, 789),
    (870, 889),
    (1114, 1027),
    (1361, 1154),
    (1642, 1288),
    (1871, 1408)
]

calculate_distance(test, h, scale.scale) # 28.0

Distance between point 0 and point 1: 3.0826033985412202 meters
Distance between point 1 and point 2: 4.218510489390111 meters
Distance between point 2 and point 3: 3.6312084170069694 meters
Distance between point 3 and point 4: 3.3264388226001844 meters
Distance between point 4 and point 5: 3.9817215185821246 meters
Distance between point 5 and point 6: 3.910880741881206 meters
Distance between point 6 and point 7: 2.7317464470004844 meters
Distance between point 7 and point 8: 2.2835432678101144 meters
Distance between point 8 and point 9: 1.562852370843475 meters
Total distance: 28.729505473655887


#### Case 2: 15.0 meters with 20 points


![Image](../../assets/homography_test_2.png)

In [9]:
test = [
    [
        2027,
        538
    ],
    [
        1971,
        555
    ],
    [
        1892,
        568
    ],
    [
        1833,
        581
    ],
    [
        1766,
        591
    ],
    [
        1719,
        606
    ],
    [
        1640,
        614
    ],
    [
        1578,
        630
    ],
    [
        1485,
        639
    ],
    [
        1435,
        656
    ],
    [
        1359,
        663
    ],
    [
        1290,
        679
    ],
    [
        1208,
        698
    ],
    [
        1129,
        709
    ],
    [
        1030,
        727
    ],
    [
        947,
        743
    ],
    [
        826,
        763
    ],
    [
        720,
        783
    ],
    [
        630,
        800
    ],
    [
        556,
        812
    ]
]

calculate_distance(test, h, scale.scale) # 15.0

Distance between point 0 and point 1: 1.0685387225768093 meters
Distance between point 1 and point 2: 0.9839845195423434 meters
Distance between point 2 and point 3: 0.8214727041002488 meters
Distance between point 3 and point 4: 0.7541564673413107 meters
Distance between point 4 and point 5: 0.7876725228065781 meters
Distance between point 5 and point 6: 0.7772333804501911 meters
Distance between point 6 and point 7: 0.8267985672515016 meters
Distance between point 7 and point 8: 0.8702958763790059 meters
Distance between point 8 and point 9: 0.7582972695341952 meters
Distance between point 9 and point 10: 0.679011031052784 meters
Distance between point 10 and point 11: 0.7486789774895711 meters
Distance between point 11 and point 12: 0.8492837182047365 meters
Distance between point 12 and point 13: 0.6703821542232278 meters
Distance between point 13 and point 14: 0.8601837799369964 meters
Distance between point 14 and point 15: 0.7056380988706338 meters
Distance between point 15 and 

#### Case 3: 26.58 meters with 6 points

![Image](../../assets/homography_test_6.png)

In [10]:
test = [
    [
        1669,
        1491
    ],
    [
        2526,
        1119
    ],
    [
        1791,
        886
    ],
    [
        2293,
        741
    ],
    [
        3036,
        895
    ],
    [
        3398,
        737
    ]
]

calculate_distance(test, h, scale.scale) # 28.0

Distance between point 0 and point 1: 5.087923221400159 meters
Distance between point 1 and point 2: 5.79893914861342 meters
Distance between point 2 and point 3: 4.8274630165299675 meters
Distance between point 3 and point 4: 5.865554494301578 meters
Distance between point 4 and point 5: 4.963325338139458 meters
Total distance: 26.543205218984582


#### Case 4: (same trajectory than use case 3) 26.58 meters with 90 points

![Image](../../assets/homography_test_5.png)

In [11]:
test = [
    [
        1672,
        1487
    ],
    [
        1723,
        1471
    ],
    [
        1754,
        1460
    ],
    [
        1801,
        1443
    ],
    [
        1831,
        1430
    ],
    [
        1878,
        1410
    ],
    [
        1922,
        1388
    ],
    [
        1976,
        1361
    ],
    [
        2043,
        1325
    ],
    [
        2086,
        1311
    ],
    [
        2113,
        1302
    ],
    [
        2155,
        1284
    ],
    [
        2207,
        1264
    ],
    [
        2231,
        1255
    ],
    [
        2270,
        1234
    ],
    [
        2293,
        1223
    ],
    [
        2346,
        1203
    ],
    [
        2375,
        1191
    ],
    [
        2411,
        1173
    ],
    [
        2442,
        1158
    ],
    [
        2483,
        1137
    ],
    [
        2508,
        1128
    ],
    [
        2527,
        1120
    ],
    [
        2525,
        1116
    ],
    [
        2498,
        1103
    ],
    [
        2469,
        1095
    ],
    [
        2433,
        1089
    ],
    [
        2390,
        1069
    ],
    [
        2354,
        1053
    ],
    [
        2296,
        1041
    ],
    [
        2240,
        1028
    ],
    [
        2185,
        1020
    ],
    [
        2148,
        1013
    ],
    [
        2152,
        1005
    ],
    [
        2118,
        990
    ],
    [
        2087,
        976
    ],
    [
        2049,
        960
    ],
    [
        1970,
        943
    ],
    [
        1923,
        933
    ],
    [
        1902,
        927
    ],
    [
        1845,
        912
    ],
    [
        1825,
        897
    ],
    [
        1801,
        887
    ],
    [
        1812,
        865
    ],
    [
        1850,
        852
    ],
    [
        1879,
        849
    ],
    [
        1929,
        835
    ],
    [
        1944,
        828
    ],
    [
        1991,
        811
    ],
    [
        2014,
        804
    ],
    [
        2055,
        797
    ],
    [
        2087,
        791
    ],
    [
        2127,
        778
    ],
    [
        2152,
        772
    ],
    [
        2186,
        767
    ],
    [
        2235,
        757
    ],
    [
        2259,
        744
    ],
    [
        2288,
        735
    ],
    [
        2321,
        737
    ],
    [
        2376,
        750
    ],
    [
        2424,
        761
    ],
    [
        2483,
        767
    ],
    [
        2513,
        775
    ],
    [
        2540,
        784
    ],
    [
        2570,
        792
    ],
    [
        2630,
        803
    ],
    [
        2688,
        811
    ],
    [
        2727,
        822
    ],
    [
        2754,
        829
    ],
    [
        2802,
        839
    ],
    [
        2826,
        843
    ],
    [
        2852,
        846
    ],
    [
        2872,
        861
    ],
    [
        2894,
        868
    ],
    [
        2929,
        868
    ],
    [
        2965,
        875
    ],
    [
        3000,
        884
    ],
    [
        3026,
        886
    ],
    [
        3077,
        872
    ],
    [
        3122,
        849
    ],
    [
        3144,
        841
    ],
    [
        3181,
        829
    ],
    [
        3213,
        812
    ],
    [
        3235,
        800
    ],
    [
        3275,
        782
    ],
    [
        3298,
        775
    ],
    [
        3330,
        768
    ],
    [
        3362,
        758
    ],
    [
        3385,
        749
    ],
    [
        3409,
        732
    ],
    [
        3411,
        732
    ]
]

calculate_distance(test, h, scale.scale) # 26.58

Distance between point 0 and point 1: 0.2122437702446145 meters
Distance between point 1 and point 2: 0.13576514303010812 meters
Distance between point 2 and point 3: 0.21073891533782715 meters
Distance between point 3 and point 4: 0.14693668357693118 meters
Distance between point 4 and point 5: 0.23332105615024726 meters
Distance between point 5 and point 6: 0.2427369052252026 meters
Distance between point 6 and point 7: 0.30794004922584367 meters
Distance between point 7 and point 8: 0.41567060734958244 meters
Distance between point 8 and point 9: 0.2185706306964985 meters
Distance between point 9 and point 10: 0.1405479262417705 meters
Distance between point 10 and point 11: 0.2479637060040237 meters
Distance between point 11 and point 12: 0.2997368628006094 meters
Distance between point 12 and point 13: 0.13963022996672908 meters
Distance between point 13 and point 14: 0.28137682134916536 meters
Distance between point 14 and point 15: 0.15829623387070904 meters
Distance between poi

### test.mp4

In [12]:
# initialize component for the homography
bd2 = BaselineDetection(
    frame_points_path="../data/data/test_points.json",
    schema_points_path="../data/data/points_cropped_schema.json"
)
h, inv_h = bd2.calculate_homography()

bd2.assess_homography_quality(h)

start_pt_1 = (527, 1279)
end_pt_1 = (3347, 1282) # Distance: 28.0m

start_pt_2 = (1935, 956)
end_pt_2 = (1933, 1317) # Distance: 15.0m


calculate_distance([start_pt_1, start_pt_2], h, scale.scale) # 28.0

print()

calculate_distance([end_pt_1, end_pt_2], h, scale.scale) # 15.0

Point 0: Error = 4.04
Point 1: Error = 3.65
Point 2: Error = 2.02
Point 3: Error = 1.60
Point 4: Error = 0.61
Point 5: Error = 5.27
Point 6: Error = 0.74
Point 7: Error = 0.27
Point 8: Error = 0.31
Point 9: Error = 1.99
Point 10: Error = 1.65
Point 11: Error = 1.93
Point 12: Error = 1.72
Point 13: Error = 4.04
Point 14: Error = 0.11
Point 15: Error = 1.39
Point 16: Error = 0.44
Point 17: Error = 3.08
Average reprojection error: 1.94
Distance between point 0 and point 1: 19.63735792928793 meters
Total distance: 19.63735792928793

Distance between point 0 and point 1: 13.973975823273715 meters
Total distance: 13.973975823273715


#### Case 5: 28.0 meters with 10 points

![Image](../../assets/homography_test_4.png)

In [13]:
# example: calculate distance of a trajectory with more points
test = [
    [
        726,
        1145
    ],
    [
        834,
        1146
    ],
    [
        914,
        1141
    ],
    [
        1039,
        1149
    ],
    [
        1140,
        1149
    ],
    [
        1232,
        1148
    ],
    [
        1369,
        1147
    ],
    [
        1467,
        1147
    ],
    [
        1578,
        1147
    ],
    [
        1667,
        1153
    ],
    [
        1799,
        1159
    ],
    [
        1908,
        1159
    ],
    [
        2021,
        1159
    ],
    [
        2143,
        1159
    ],
    [
        2257,
        1157
    ],
    [
        2358,
        1152
    ],
    [
        2462,
        1147
    ],
    [
        2587,
        1143
    ],
    [
        2672,
        1146
    ],
    [
        2806,
        1147
    ],
    [
        2908,
        1151
    ],
    [
        2969,
        1147
    ],
    [
        3024,
        1149
    ],
    [
        3108,
        1147
    ],
    [
        3146,
        1149
    ]
]

calculate_distance(test, h, scale.scale) # 28.0


Distance between point 0 and point 1: 1.2589215428744505 meters
Distance between point 1 and point 2: 0.8674355549536751 meters
Distance between point 2 and point 3: 1.5782221707477504 meters
Distance between point 3 and point 4: 1.1572812661341811 meters
Distance between point 4 and point 5: 1.045234904843483 meters
Distance between point 5 and point 6: 1.5652387121137714 meters
Distance between point 6 and point 7: 1.1268233297051111 meters
Distance between point 7 and point 8: 1.2766929096738795 meters
Distance between point 8 and point 9: 1.0696037535237672 meters
Distance between point 9 and point 10: 1.5339992920741023 meters
Distance between point 10 and point 11: 1.2361973894680078 meters
Distance between point 11 and point 12: 1.281975630318195 meters
Distance between point 12 and point 13: 1.3845523522052372 meters
Distance between point 13 and point 14: 1.305895989948029 meters
Distance between point 14 and point 15: 1.1958282451788937 meters
Distance between point 15 and po

#### Case 6: 15.0 meters with 10 points

![Image](../../assets/homography_test_3.png)

In [14]:
test = [
    (1933, 955),
    (1934, 984),
    (1934, 1032),
    (1934, 1080),
    (1934, 1126),
    (1934, 1171),
    (1934, 1258),
    (1937, 1283),
    (1934, 1316)
]

calculate_distance(test, h, scale.scale) # 15.0

Distance between point 0 and point 1: 1.8240660301198484 meters
Distance between point 1 and point 2: 2.6760608223478983 meters
Distance between point 2 and point 3: 2.3219331720530603 meters
Distance between point 3 and point 4: 1.954050394099949 meters
Distance between point 4 and point 5: 1.6985108337565171 meters
Distance between point 5 and point 6: 2.8040044996531805 meters
Distance between point 6 and point 7: 0.7089907440740819 meters
Distance between point 7 and point 8: 0.8795314882521885 meters
Total distance: 14.867147984356725
